# ระบบจำแนกคำร้องมหาวิทยาลัยด้วย Typhoon + QLoRA
## Advanced Training, Evaluation, Calibration, Threshold และ Word Contribution

Notebook นี้พัฒนาต่อจาก `new.ipynb` และคง Path เดิมที่ `D:\model\new`

**เป้าหมายหลัก**
- Fine-tune `typhoon-ai/typhoon2.5-qwen3-4b` ด้วย 4-bit QLoRA
- แยก Train / Validation / Test
- คำนวณ Loss เฉพาะ Completion ไม่รวม System/User prompt
- ประเมิน Accuracy, Precision, Recall, Macro-F1, Weighted-F1 และ Confusion Matrix
- Calibrate Confidence ด้วย Temperature Scaling
- หา Threshold ที่เหมาะจาก Validation และปรับเองได้ 1%–99%
- ใช้ Top-1 / Top-2 / Margin ช่วยจับข้อความกำกวม/หลายประเด็น
- วิเคราะห์อิทธิพลแต่ละคำด้วย Word Occlusion Contribution
- มี Baseline TF-IDF + Logistic Regression
- มี Optional Hyperparameter Sweep สำหรับ Learning Rate โดยเริ่มจาก Base Model ใหม่ทุกครั้ง

> Confidence ของ Generative LLM เป็น **normalized candidate confidence** ไม่ใช่ probability ที่ calibrated โดยธรรมชาติ จึงต้องมี Calibration ก่อนใช้ Threshold จริง

## แนวทางอ้างอิง
- TRL SFTTrainer / completion-only loss: https://huggingface.co/docs/trl/main/sft_trainer
- QLoRA / NF4: https://huggingface.co/docs/peft/developer_guides/quantization
- Temperature scaling: https://proceedings.mlr.press/v70/guo17a.html
- Selective classification: https://arxiv.org/abs/1705.08500
- Student complaint classification: https://ijaims.smiu.edu.pk/index.php/AIMS/article/view/135
- Dataset ต้นทาง: https://huggingface.co/datasets/alaminxpro/university-students-complaints

In [ ]:
# ============================================================
# CELL 1 : ติดตั้ง Library ที่ยังไม่มี
# ============================================================
import importlib, subprocess, sys

REQUIRED = {
    "torch": "torch", "transformers": "transformers", "datasets": "datasets",
    "peft": "peft", "bitsandbytes": "bitsandbytes", "trl": "trl",
    "sklearn": "scikit-learn", "pandas": "pandas", "numpy": "numpy",
    "matplotlib": "matplotlib", "scipy": "scipy", "pythainlp": "pythainlp",
    "ipywidgets": "ipywidgets"
}
missing=[]
for mod,pkg in REQUIRED.items():
    try: importlib.import_module(mod)
    except ImportError: missing.append(pkg)
if missing:
    print("กำลังติดตั้ง:", missing)
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
else:
    print("Library ที่จำเป็นมีครบแล้ว")

In [ ]:
# ============================================================
# CELL 2 : Import + Versions
# ============================================================
import os, gc, re, json, math, random, warnings, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers, datasets, peft, trl
from datasets import Dataset
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTConfig, SFTTrainer
warnings.filterwarnings("ignore")
print("Python:",sys.version.split()[0])
print("PyTorch:",torch.__version__)
print("Transformers:",transformers.__version__)
print("Datasets:",datasets.__version__)
print("PEFT:",peft.__version__)
print("TRL:",trl.__version__)

In [ ]:
# ============================================================
# CELL 3 : PATH + CONFIG ทั้งหมด (คง Path เดิม)
# ============================================================
BASE_DIR = Path(r"D:\model\new")
DATASET_PATH = BASE_DIR / "dataset.json"
OUTPUT_DIR = BASE_DIR / "typhoon-training-output"
EVAL_DIR = BASE_DIR / "evaluation"
EXPERIMENT_DIR = BASE_DIR / "experiments"
for p in [BASE_DIR,OUTPUT_DIR,EVAL_DIR,EXPERIMENT_DIR]: p.mkdir(parents=True,exist_ok=True)

MODEL_ID = "typhoon-ai/typhoon2.5-qwen3-4b"
SEED=42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

TRAIN_EXPERIMENTS = {
    "E1_lr5e-5": {"learning_rate":5e-5,"epochs":3},
    "E2_lr1e-4": {"learning_rate":1e-4,"epochs":3},
    "E3_lr2e-4": {"learning_rate":2e-4,"epochs":3},
}
SELECTED_EXPERIMENT="E2_lr1e-4"
RUN_TRAINING=True
RUN_HPARAM_SWEEP=False

MAX_LENGTH=256          # ลดจาก 512 → 256 ประหยัด VRAM บน RTX 4050 (6.4 GB)
TRAIN_BATCH_SIZE=1
EVAL_BATCH_SIZE=1
GRAD_ACCUM=8
USE_FP16=True           # RTX 4050 ไม่รองรับ bf16 → ต้องใช้ fp16
USE_BF16=False          # bf16 ใช้ได้เฉพาะ A100/H100

MANUAL_THRESHOLD=0.75
MARGIN_THRESHOLD=0.10
SECONDARY_THRESHOLD=0.20
TARGET_SELECTIVE_ACCURACY=0.90
MIN_AUTO_COVERAGE=0.30
MIN_ACCEPTED_SAMPLES=5

print("DATASET_PATH:",DATASET_PATH)
print("OUTPUT_DIR:",OUTPUT_DIR)
print("Experiment:",SELECTED_EXPERIMENT,TRAIN_EXPERIMENTS[SELECTED_EXPERIMENT])
print(f"USE_FP16={USE_FP16}, USE_BF16={USE_BF16}, MAX_LENGTH={MAX_LENGTH}")

In [ ]:
# ============================================================
# CELL 4 : Seed + GPU
# ============================================================
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(SEED)
print("CUDA Available:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))
    print("CUDA:",torch.version.cuda)
    print("VRAM GB:",round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))

In [ ]:
# ============================================================
# CELL 5 : โหลด Dataset + Parse label/category
# ============================================================
assert DATASET_PATH.exists(), f"ไม่พบไฟล์: {DATASET_PATH}"
with open(DATASET_PATH,"r",encoding="utf-8") as f: raw=json.load(f)
df=pd.DataFrame(raw)
assert {"input","output"}.issubset(df.columns), "dataset.json ต้องมี input และ output"

def parse_output(text):
    text=str(text).strip()
    lm=re.search(r"label\s*:\s*(\d+)",text,re.I)
    cm=re.search(r"category\s*:\s*(.+)",text,re.I)
    return (int(lm.group(1)) if lm else None, cm.group(1).strip() if cm else None)
parsed=df["output"].apply(parse_output)
df["label"]=parsed.apply(lambda x:x[0])
df["category"]=parsed.apply(lambda x:x[1])
bad=df[df["label"].isna()|df["category"].isna()]
if len(bad):
    display(bad); raise ValueError("มี output ที่ parse ไม่ได้")
df["label"]=df["label"].astype(int)
df["input"]=df["input"].astype(str).str.strip()
CATEGORY_MAP=(df[["label","category"]].drop_duplicates().sort_values("label")
              .set_index("label")["category"].to_dict())
print("จำนวนข้อมูล:",len(df))
for k,v in CATEGORY_MAP.items(): print(k,"->",v)
display(df.head())

In [ ]:
# ============================================================
# CELL 6 : Data Quality / Duplicate / Class Balance
# ============================================================
def normalize_text(t):
    t=str(t).strip().lower(); t=re.sub(r"\s+","",t); t=re.sub(r"[^\wก-๙]","",t); return t

df["text_normalized"]=df["input"].apply(normalize_text)
print("Exact duplicate:",df.duplicated("input").sum())
print("Normalized duplicate:",df.duplicated("text_normalized").sum())
class_table=df.groupby(["label","category"]).size().reset_index(name="count")
class_table["percent"]=class_table["count"]/len(df)*100
display(class_table)
GROUP_CANDIDATES=["Complaint_Group_ID","complaint_group_id","group_id","Group_ID"]
GROUP_COLUMN=next((c for c in GROUP_CANDIDATES if c in df.columns),None)
print("Group column:",GROUP_COLUMN)
if GROUP_COLUMN is None:
    print("⚠️ ไม่มี Group ID: ยังป้องกัน paraphrase leakage ได้ไม่สมบูรณ์")

In [ ]:
# ============================================================
# CELL 7 : Train / Validation / Test Split
# ============================================================
def stratified_split(data):
    strat1=data["label"] if data["label"].value_counts().min()>=2 else None
    train_val,test=train_test_split(data,test_size=TEST_RATIO,random_state=SEED,stratify=strat1)
    rel_val=VAL_RATIO/(TRAIN_RATIO+VAL_RATIO)
    strat2=train_val["label"] if train_val["label"].value_counts().min()>=2 else None
    train,val=train_test_split(train_val,test_size=rel_val,random_state=SEED,stratify=strat2)
    return train,val,test

def grouped_split(data,col):
    s1=GroupShuffleSplit(n_splits=1,test_size=TEST_RATIO,random_state=SEED)
    a,b=next(s1.split(data,groups=data[col])); train_val=data.iloc[a].copy(); test=data.iloc[b].copy()
    rel_val=VAL_RATIO/(TRAIN_RATIO+VAL_RATIO)
    s2=GroupShuffleSplit(n_splits=1,test_size=rel_val,random_state=SEED+1)
    a,b=next(s2.split(train_val,groups=train_val[col])); return train_val.iloc[a].copy(),train_val.iloc[b].copy(),test

if GROUP_COLUMN:
    train_df,val_df,test_df=grouped_split(df,GROUP_COLUMN); SPLIT_METHOD=f"Group split: {GROUP_COLUMN}"
else:
    train_df,val_df,test_df=stratified_split(df); SPLIT_METHOD="Stratified split"
for x in [train_df,val_df,test_df]: x.reset_index(drop=True,inplace=True)
print(SPLIT_METHOD)
print("Train",len(train_df),"Validation",len(val_df),"Test",len(test_df))
dist=pd.concat([train_df.groupby("label").size().rename("Train"),val_df.groupby("label").size().rename("Validation"),test_df.groupby("label").size().rename("Test")],axis=1).fillna(0).astype(int)
dist["Category"]=[CATEGORY_MAP.get(i,"?") for i in dist.index]
display(dist)
print("Dup Train-Val",len(set(train_df.text_normalized)&set(val_df.text_normalized)))
print("Dup Train-Test",len(set(train_df.text_normalized)&set(test_df.text_normalized)))

In [ ]:
# ============================================================
# CELL 8 : Baseline TF-IDF + Logistic Regression
# ============================================================
baseline=Pipeline([
    ("tfidf",TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=1,sublinear_tf=True)),
    ("clf",LogisticRegression(max_iter=3000,class_weight="balanced",random_state=SEED))
])
baseline.fit(train_df["input"],train_df["label"])
baseline_pred=baseline.predict(test_df["input"])
baseline_acc=accuracy_score(test_df["label"],baseline_pred)
baseline_macro_f1=f1_score(test_df["label"],baseline_pred,average="macro",zero_division=0)
print("Baseline Accuracy:",round(baseline_acc,4))
print("Baseline Macro-F1:",round(baseline_macro_f1,4))

In [ ]:
# ============================================================
# CELL 9 : System Prompt จาก Label จริง
# ============================================================
category_lines="\n".join(f"{k} = {v}" for k,v in CATEGORY_MAP.items())
SYSTEM_PROMPT=f"""คุณคือระบบจำแนกคำร้องของมหาวิทยาลัย

เลือกเพียง 1 หมวดหลักจากรายการนี้:
{category_lines}

กติกา:
1. พิจารณาความหมายโดยรวม ไม่ใช้ keyword เพียงคำเดียว
2. ถ้ามีหลายเรื่อง ให้เลือกปัญหาหลักที่สุด
3. ห้ามสร้าง label/category นอกรายการ
4. ตอบเพียง: label: <เลข>, category: <ชื่อหมวดหมู่>
5. ห้ามอธิบายเพิ่มเติม""".strip()
print(SYSTEM_PROMPT)

In [ ]:
# ============================================================
# CELL 10 : Prompt-Completion Dataset (Loss เฉพาะคำตอบ)
# ============================================================
def to_record(row):
    return {
        "prompt":[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":f"ข้อความ: {row['input']}"}],
        "completion":[{"role":"assistant","content":row["output"]}],
        "input_text":row["input"],"true_label":int(row["label"]),"true_category":row["category"]
    }
train_records=[to_record(x) for x in train_df.to_dict("records")]
val_records=[to_record(x) for x in val_df.to_dict("records")]
test_records=[to_record(x) for x in test_df.to_dict("records")]
train_ds, val_ds, test_ds = Dataset.from_list(train_records),Dataset.from_list(val_records),Dataset.from_list(test_records)
print(train_ds); print(val_ds); print(test_ds)

In [ ]:
# ============================================================
# CELL 11 : QLoRA Model Factory / Load Saved Adapter
# ============================================================
def load_tokenizer(source=MODEL_ID):
    tok=AutoTokenizer.from_pretrained(source); tok.padding_side="right"
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    return tok

def create_quantized_base_model():
    qconfig=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    kwargs=dict(
        pretrained_model_name_or_path=MODEL_ID,
        quantization_config=qconfig,
        device_map="auto",
    )
    try:
        m=AutoModelForCausalLM.from_pretrained(**kwargs,dtype=torch.float16)
    except TypeError:
        m=AutoModelForCausalLM.from_pretrained(**kwargs,torch_dtype=torch.float16)
    m.config.use_cache=False
    return m

def create_fresh_qlora_model():
    m=create_quantized_base_model()
    m=prepare_model_for_kbit_training(m,use_gradient_checkpointing=True)
    lcfg=LoraConfig(
        r=16,lora_alpha=32,lora_dropout=0.05,bias="none",
        task_type="CAUSAL_LM",target_modules="all-linear"
    )
    return get_peft_model(m,lcfg)

def load_saved_adapter(adapter_dir=OUTPUT_DIR):
    adapter_dir=Path(adapter_dir)
    if not (adapter_dir / "adapter_config.json").exists():
        raise FileNotFoundError(
            f"ไม่พบ LoRA adapter ที่ {adapter_dir}\n"
            "ให้ตั้ง RUN_TRAINING=True เพื่อ Train ก่อน"
        )
    base=create_quantized_base_model()
    m=PeftModel.from_pretrained(base,str(adapter_dir),is_trainable=False)
    m.eval()
    return m

tokenizer=load_tokenizer()
print("Tokenizer พร้อม")

In [ ]:
# ============================================================
# CELL 12 : Token Length Audit
# ============================================================
def count_tokens(rec):
    txt=tokenizer.apply_chat_template(rec["prompt"]+rec["completion"],tokenize=False)
    return len(tokenizer(txt,add_special_tokens=False)["input_ids"])
lengths=[count_tokens(x) for x in train_records]
print("Min",min(lengths),"Median",int(np.median(lengths)),"P95",int(np.percentile(lengths,95)),"Max",max(lengths))
print("เกิน MAX_LENGTH:",sum(x>MAX_LENGTH for x in lengths),"จาก",len(lengths))

In [ ]:
# ============================================================
# CELL 13 : Training Config
# ============================================================
exp = TRAIN_EXPERIMENTS[SELECTED_EXPERIMENT]

# คำนวณ warmup_steps แทน warmup_ratio (trl 1.x ลบ warmup_ratio ออกจาก SFTConfig)
total_steps = (len(train_ds) // (TRAIN_BATCH_SIZE * GRAD_ACCUM)) * exp["epochs"]
warmup_steps = max(1, int(total_steps * 0.05))

sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=exp["epochs"],
    learning_rate=exp["learning_rate"],
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    logging_steps=5,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=USE_FP16,
    bf16=USE_BF16,
    completion_only_loss=True,
    max_length=MAX_LENGTH,
    packing=False,
    report_to="none",
    seed=SEED,
)
print("LR", exp["learning_rate"], "Epochs", exp["epochs"])
print(f"warmup_steps = {warmup_steps} (จาก {total_steps} total steps)")

In [ ]:
# ============================================================
# CELL 14 : โหลด Main Model + Trainer หรือ Adapter เดิม
# ============================================================
if RUN_TRAINING:
    model=create_fresh_qlora_model()
    model.print_trainable_parameters()
    trainer=SFTTrainer(
        model=model,args=sft_args,train_dataset=train_ds,
        eval_dataset=val_ds,processing_class=tokenizer
    )
    print("Trainer พร้อมสำหรับ Train")
else:
    print("RUN_TRAINING=False -> โหลด Adapter ที่เคย Train ไว้")
    # ถ้า OUTPUT_DIR มี tokenizer ที่ save ไว้ ให้ใช้ tokenizer นั้น
    if (OUTPUT_DIR / "tokenizer_config.json").exists():
        tokenizer=load_tokenizer(str(OUTPUT_DIR))
    model=load_saved_adapter(OUTPUT_DIR)
    trainer=None
    print("โหลด Adapter สำเร็จ:",OUTPUT_DIR)

In [ ]:
# ============================================================
# CELL 15 : Train + Save
# ============================================================
if RUN_TRAINING:
    train_result=trainer.train()
    print(train_result)
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    model.eval()
    print("บันทึกที่",OUTPUT_DIR)
else:
    print("ข้ามการ Train และใช้ Adapter ที่โหลดจาก OUTPUT_DIR")

In [ ]:
# ============================================================
# CELL 16 : Training / Validation Loss
# ============================================================
if trainer is None:
    print("โหมด Load Adapter: ไม่มี log_history ของการ Train ใน session นี้")
    print("หากต้องการกราฟ ให้รัน Train ใน Notebook นี้ หรือเก็บ trainer_state.json จากรอบ Train")
else:
    logs=trainer.state.log_history
    train_logs=[x for x in logs if "loss" in x and "eval_loss" not in x]
    eval_logs=[x for x in logs if "eval_loss" in x]
    if train_logs:
        tdf=pd.DataFrame(train_logs); display(tdf)
        plt.figure(figsize=(10,4)); plt.plot(tdf["step"],tdf["loss"],marker="o"); plt.xlabel("Step"); plt.ylabel("Training Loss"); plt.grid(alpha=.3); plt.show()
    if eval_logs:
        edf=pd.DataFrame(eval_logs); display(edf)
        plt.figure(figsize=(10,4)); plt.plot(edf["epoch"],edf["eval_loss"],marker="o"); plt.xlabel("Epoch"); plt.ylabel("Validation Loss"); plt.grid(alpha=.3); plt.show()

## Optional Hyperparameter Sweep
ตั้ง `RUN_HPARAM_SWEEP=True` หากต้องการเทรน E1/E2/E3 แยกกัน โดยต้องเริ่มจาก Base Model ใหม่ทุกครั้ง  
**แนะนำ Restart Kernel ก่อนรัน Sweep** เพื่อลดโอกาส VRAM ค้าง และอย่าใช้ Test set เลือกค่า training

In [ ]:
# ============================================================
# CELL 17 : ฟังก์ชัน Generate Label
# ============================================================
def get_input_device(m): return m.get_input_embeddings().weight.device

@torch.inference_mode()
def generate_label(m,tok,text,max_new_tokens=40):
    messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":f"ข้อความ: {text}"}]
    prompt=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tok(prompt,return_tensors="pt"); device=get_input_device(m); inputs={k:v.to(device) for k,v in inputs.items()}
    n=inputs["input_ids"].shape[1]
    out=m.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tok.pad_token_id,eos_token_id=tok.eos_token_id)
    ans=tok.decode(out[0][n:],skip_special_tokens=True).strip()
    mm=re.search(r"label\s*:\s*(\d+)",ans,re.I)
    return (int(mm.group(1)) if mm else None),ans

In [ ]:
# ============================================================
# CELL 18 : Optional LR Sweep (Fresh Base Model ทุกครั้ง)
# ============================================================
sweep_results=[]
if RUN_HPARAM_SWEEP:
    print("⚠️ Sweep เป็นโหมดแยก: หลัง Sweep เสร็จให้ Restart Kernel แล้วเลือก SELECTED_EXPERIMENT ที่ดีที่สุด ก่อน Train Main model")
    try: del trainer,model
    except: pass
    gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    for name,cfg in TRAIN_EXPERIMENTS.items():
        print("\n###",name,cfg)
        outdir=EXPERIMENT_DIR/name; outdir.mkdir(parents=True,exist_ok=True)
        m=create_fresh_qlora_model(); tok=load_tokenizer()
        args=SFTConfig(output_dir=str(outdir),num_train_epochs=cfg["epochs"],learning_rate=cfg["learning_rate"],per_device_train_batch_size=TRAIN_BATCH_SIZE,per_device_eval_batch_size=EVAL_BATCH_SIZE,gradient_accumulation_steps=GRAD_ACCUM,optim="paged_adamw_8bit",weight_decay=.01,max_grad_norm=1.0,lr_scheduler_type="cosine",warmup_ratio=.05,logging_steps=5,eval_strategy="epoch",save_strategy="no",gradient_checkpointing=True,gradient_checkpointing_kwargs={"use_reentrant":False},fp16=USE_FP16,bf16=USE_BF16,completion_only_loss=True,max_length=MAX_LENGTH,packing=False,report_to="none",seed=SEED)
        tr=SFTTrainer(model=m,args=args,train_dataset=train_ds,eval_dataset=val_ds,processing_class=tok)
        tr.train(); ev=tr.evaluate(); yt=[]; yp=[]
        for row in val_df.to_dict("records"):
            p,_=generate_label(m,tok,row["input"])
            if p is not None: yt.append(int(row["label"])); yp.append(p)
        acc=accuracy_score(yt,yp) if yt else np.nan
        mf1=f1_score(yt,yp,average="macro",zero_division=0) if yt else np.nan
        tr.save_model(str(outdir)); tok.save_pretrained(str(outdir))
        sweep_results.append({"experiment":name,"learning_rate":cfg["learning_rate"],"eval_loss":ev.get("eval_loss"),"val_accuracy":acc,"val_macro_f1":mf1})
        del tr,m,tok; gc.collect();
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    sweep_df=pd.DataFrame(sweep_results); sweep_df.to_csv(EXPERIMENT_DIR/"sweep_results.csv",index=False,encoding="utf-8-sig")
    display(sweep_df.sort_values(["val_macro_f1","eval_loss"],ascending=[False,True]))
else:
    print("RUN_HPARAM_SWEEP=False")

## Candidate Confidence
เพราะโมเดลเป็น Causal LM จึงไม่มี 8-class softmax head โดยตรง เราจะคำนวณ likelihood ของคำตอบแต่ละหมวด แล้ว Softmax ข้าม candidate ทั้งหมด จากนั้นทำ Temperature Scaling บน Validation

In [ ]:
# ============================================================
# CELL 19 : Candidate Likelihood Scoring
# ============================================================
def build_prompt(text):
    msgs=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":f"ข้อความ: {text}"}]
    return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)

@torch.inference_mode()
def score_candidate(text,label,category):
    prompt=build_prompt(text); candidate=f"label: {label}, category: {category}"
    pids=tokenizer(prompt,add_special_tokens=False,return_tensors="pt")["input_ids"][0]
    cids=tokenizer(candidate,add_special_tokens=False,return_tensors="pt")["input_ids"][0]
    ids=torch.cat([pids,cids]).unsqueeze(0); mask=torch.ones_like(ids); dev=get_input_device(model); ids=ids.to(dev); mask=mask.to(dev)
    logits=model(input_ids=ids,attention_mask=mask).logits
    plen=len(pids); clen=len(cids)
    pred=logits[0,plen-1:plen+clen-1,:]
    lp=torch.log_softmax(pred,dim=-1); target=cids.to(dev)
    token_lp=lp.gather(1,target.unsqueeze(1)).squeeze(1)
    return float(token_lp.mean().item())

def get_raw_scores(text):
    return [{"label":int(k),"category":v,"raw_score":score_candidate(text,k,v)} for k,v in CATEGORY_MAP.items()]

def stable_softmax(x):
    x=np.asarray(x,dtype=np.float64); x=x-np.max(x); e=np.exp(x); return e/e.sum()

In [ ]:
# ============================================================
# CELL 20 : Validation Score Cache
# ============================================================
validation_cache=[]
for i,row in enumerate(val_df.to_dict("records"),1):
    validation_cache.append({"text":row["input"],"true_label":int(row["label"]),"raw_scores":get_raw_scores(row["input"])})
    print(f"{i}/{len(val_df)}",end="\r")
print("\nValidation cache:",len(validation_cache))

In [ ]:
# ============================================================
# CELL 21 : Temperature Scaling + ECE
# ============================================================
from scipy.optimize import minimize_scalar
LABEL_ORDER=sorted(CATEGORY_MAP); LABEL_TO_INDEX={x:i for i,x in enumerate(LABEL_ORDER)}
def probs_from_raw(raw,T=1.0): return stable_softmax(np.array([x["raw_score"] for x in raw])/max(float(T),1e-6))
def nll(T):
    losses=[]
    for item in validation_cache:
        p=probs_from_raw(item["raw_scores"],T); pt=max(float(p[LABEL_TO_INDEX[item["true_label"]]]),1e-12); losses.append(-math.log(pt))
    return float(np.mean(losses))
def ece(confs,correct,n_bins=10):
    confs=np.asarray(confs); correct=np.asarray(correct,float); bins=np.linspace(0,1,n_bins+1); out=0.0
    for a,b in zip(bins[:-1],bins[1:]):
        m=(confs>a)&(confs<=b)
        if m.sum(): out+=m.mean()*abs(correct[m].mean()-confs[m].mean())
    return float(out)
opt=minimize_scalar(nll,bounds=(0.05,10),method="bounded"); BEST_TEMPERATURE=float(opt.x)
print("Best Temperature:",round(BEST_TEMPERATURE,4),"NLL:",round(opt.fun,4))
for T,name in [(1.0,"ก่อน"),(BEST_TEMPERATURE,"หลัง")]:
    conf=[]; cor=[]
    for it in validation_cache:
        p=probs_from_raw(it["raw_scores"],T); pred=LABEL_ORDER[int(np.argmax(p))]; conf.append(float(p.max())); cor.append(int(pred==it["true_label"]))
    print(name,"Calibration ECE:",round(ece(conf,cor),4))
if len(val_df)<50: print("⚠️ Validation < 50: Calibration/Threshold ยังเป็นค่าทดลองเบื้องต้น")

In [ ]:
# ============================================================
# CELL 22 : Score DataFrame
# ============================================================
def score_dataframe(raw,T=None):
    if T is None: T=BEST_TEMPERATURE
    probs=probs_from_raw(raw,T); rows=[]
    for item,p in zip(raw,probs): rows.append({"Label":item["label"],"Category":item["category"],"Raw Score":item["raw_score"],"Confidence":float(p),"Confidence (%)":float(p*100)})
    return pd.DataFrame(rows).sort_values("Confidence",ascending=False).reset_index(drop=True)

## Threshold 1%–99%
เมื่อมี 8 หมวด ค่าเท่ากันทุกหมวดจะอยู่ประมาณ **12.5%** ดังนั้น Threshold 1% ต่ำเกินไปและแทบไม่ reject อะไรเลย  
ให้หา Threshold จาก Validation โดยดู **Coverage**, **Selective Accuracy**, **Selective Risk**, และ **Margin** แทนการเดา 75% หรือ 80%

In [ ]:
# ============================================================
# CELL 23 : Validation Predictions
# ============================================================
rows=[]
for it in validation_cache:
    s=score_dataframe(it["raw_scores"]); a=s.iloc[0]; b=s.iloc[1]
    rows.append({"text":it["text"],"true_label":it["true_label"],"pred_label":int(a["Label"]),"confidence":float(a["Confidence"]),"top2_confidence":float(b["Confidence"]),"margin":float(a["Confidence"]-b["Confidence"]),"correct":int(int(a["Label"])==it["true_label"])})
validation_pred_df=pd.DataFrame(rows); display(validation_pred_df)

In [ ]:
# ============================================================
# CELL 24 : Threshold Search 1% - 99%
# ============================================================
def wilson_lb(k,n,z=1.96):
    if n==0:return np.nan
    p=k/n; den=1+z*z/n; center=(p+z*z/(2*n))/den; margin=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den; return max(0.0,center-margin)
trs=[]
for th in np.arange(.01,1,.01):
    mask=(validation_pred_df.confidence>=th)&(validation_pred_df.margin>=MARGIN_THRESHOLD); sub=validation_pred_df[mask]; n=len(sub); total=len(validation_pred_df); cov=n/total if total else 0
    if n:
        k=int(sub.correct.sum()); acc=k/n; risk=1-acc; lb=wilson_lb(k,n)
    else: acc=risk=lb=np.nan
    trs.append({"Threshold":th,"Threshold (%)":th*100,"Accepted":n,"Coverage":cov,"Coverage (%)":cov*100,"Selective Accuracy":acc,"Selective Accuracy (%)":acc*100 if not np.isnan(acc) else np.nan,"Selective Risk (%)":risk*100 if not np.isnan(risk) else np.nan,"Wilson LB 95% (%)":lb*100 if not np.isnan(lb) else np.nan})
threshold_df=pd.DataFrame(trs)
important=[1,10,20,30,40,50,60,70,75,80,85,90,95,99]
display(threshold_df[threshold_df["Threshold (%)"].isin(important)][["Threshold (%)","Accepted","Coverage (%)","Selective Accuracy (%)","Selective Risk (%)","Wilson LB 95% (%)"]].round(2))

In [ ]:
# ============================================================
# CELL 25 : Auto Threshold Recommendation
# ============================================================
cand=threshold_df[(threshold_df.Accepted>=MIN_ACCEPTED_SAMPLES)&(threshold_df.Coverage>=MIN_AUTO_COVERAGE)&(threshold_df["Selective Accuracy"]>=TARGET_SELECTIVE_ACCURACY)].copy()
if len(cand):
    best=cand.sort_values(["Coverage","Threshold"],ascending=[False,True]).iloc[0]; AUTO_THRESHOLD=float(best.Threshold)
    print("AUTO_THRESHOLD",f"{AUTO_THRESHOLD*100:.1f}%","Coverage",f"{best['Coverage (%)']:.2f}%","Selective Accuracy",f"{best['Selective Accuracy (%)']:.2f}%","Wilson LB",f"{best['Wilson LB 95% (%)']:.2f}%")
else:
    AUTO_THRESHOLD=MANUAL_THRESHOLD; print("⚠️ ยังหา threshold ที่ผ่านเงื่อนไขไม่ได้ ใช้ manual",f"{AUTO_THRESHOLD*100:.1f}%")

In [ ]:
# ============================================================
# CELL 26 : Coverage vs Selective Accuracy
# ============================================================
p=threshold_df.dropna(subset=["Selective Accuracy"])
plt.figure(figsize=(10,5)); plt.plot(p["Threshold (%)"],p["Coverage (%)"],label="Coverage"); plt.plot(p["Threshold (%)"],p["Selective Accuracy (%)"],label="Selective Accuracy"); plt.axvline(AUTO_THRESHOLD*100,linestyle="--",label=f"Auto={AUTO_THRESHOLD*100:.0f}%"); plt.xlabel("Threshold (%)"); plt.ylabel("Percent"); plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
# ============================================================
# CELL 27 : Test Score Cache
# ============================================================
test_cache=[]
for i,row in enumerate(test_df.to_dict("records"),1):
    test_cache.append({"text":row["input"],"true_label":int(row["label"]),"raw_scores":get_raw_scores(row["input"])})
    print(f"{i}/{len(test_df)}",end="\r")
print("\nTest cache",len(test_cache))

In [ ]:
# ============================================================
# CELL 28 : Test Metrics
# ============================================================
rows=[]
for it in test_cache:
    s=score_dataframe(it["raw_scores"]); a=s.iloc[0]; b=s.iloc[1]
    rows.append({"text":it["text"],"true_label":it["true_label"],"pred_label":int(a.Label),"confidence":float(a.Confidence),"top2_label":int(b.Label),"top2_confidence":float(b.Confidence),"margin":float(a.Confidence-b.Confidence)})
test_pred_df=pd.DataFrame(rows); yt=test_pred_df.true_label; yp=test_pred_df.pred_label
acc=accuracy_score(yt,yp); mp,mr,mf1,_=precision_recall_fscore_support(yt,yp,average="macro",zero_division=0); wf1=f1_score(yt,yp,average="weighted",zero_division=0)
metrics_summary=pd.DataFrame([{"Accuracy":acc,"Macro Precision":mp,"Macro Recall":mr,"Macro F1":mf1,"Weighted F1":wf1,"Baseline Accuracy":baseline_acc,"Baseline Macro F1":baseline_macro_f1}]); display(metrics_summary.round(4))
print(classification_report(yt,yp,labels=LABEL_ORDER,target_names=[CATEGORY_MAP[i] for i in LABEL_ORDER],digits=4,zero_division=0))
mask=(test_pred_df.confidence>=AUTO_THRESHOLD)&(test_pred_df.margin>=MARGIN_THRESHOLD); accepted=test_pred_df[mask]; coverage=len(accepted)/len(test_pred_df); sacc=accuracy_score(accepted.true_label,accepted.pred_label) if len(accepted) else np.nan
print("Threshold",f"{AUTO_THRESHOLD*100:.1f}%","Coverage",f"{coverage*100:.2f}%","Selective Accuracy",f"{sacc*100:.2f}%" if not np.isnan(sacc) else "N/A")

In [ ]:
# ============================================================
# CELL 29 : Confusion Matrix
# ============================================================
cm=confusion_matrix(yt,yp,labels=LABEL_ORDER); fig,ax=plt.subplots(figsize=(10,8)); ConfusionMatrixDisplay(cm,display_labels=[CATEGORY_MAP[i] for i in LABEL_ORDER]).plot(ax=ax,xticks_rotation=45,values_format="d"); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# CELL 30 : ฟังก์ชันวิเคราะห์ข้อความ / Top1 Top2 Margin
# ============================================================
def analyze_text(text,threshold=None,margin_threshold=MARGIN_THRESHOLD,secondary_threshold=SECONDARY_THRESHOLD,verbose=True):
    if threshold is None: threshold=AUTO_THRESHOLD
    raw=get_raw_scores(text); s=score_dataframe(raw); a=s.iloc[0]; b=s.iloc[1]
    c1=float(a.Confidence); c2=float(b.Confidence); margin=c1-c2
    multi=(c2>=secondary_threshold) or (margin<margin_threshold)
    if c1>=threshold and margin>=margin_threshold: status="✅ รับอัตโนมัติ"
    elif c1<threshold: status="⚠️ Confidence ต่ำ → ให้ผู้ดูแลตรวจ"
    else: status="⚠️ Top-1/Top-2 ใกล้กัน → ให้ผู้ดูแลตรวจ"
    out={"text":text,"threshold":threshold,"top1_label":int(a.Label),"top1_category":a.Category,"top1_confidence":c1,"top2_label":int(b.Label),"top2_category":b.Category,"top2_confidence":c2,"margin":margin,"possible_multi_issue":multi,"status":status,"score_df":s}
    if verbose:
        print("ข้อความ:",text); display(s[["Label","Category","Raw Score","Confidence (%)"]].round({"Raw Score":4,"Confidence (%)":2})); print("Top-1",a.Category,f"{c1*100:.2f}%"); print("Top-2",b.Category,f"{c2*100:.2f}%"); print("Margin",f"{margin*100:.2f} pp"); print("Threshold",f"{threshold*100:.2f}%"); print(status); print("🔶 อาจมีหลายประเด็น/กำกวม" if multi else "🟢 แยกอันดับค่อนข้างชัด")
    return out

In [ ]:
# ============================================================
# CELL 31 : ⭐ พิมพ์ข้อความ Test ตรงนี้
# ============================================================
TEST_TEXT="รถเมย์ชนหมาเลือดสาดแม่บ้านมาทำความสะอาดหน่อย"
TEST_THRESHOLD=0.75      # ทดลอง 0.01 / 0.50 / 0.75 / 0.80 / 0.90
USE_AUTO_THRESHOLD=False
threshold_to_use=AUTO_THRESHOLD if USE_AUTO_THRESHOLD else TEST_THRESHOLD
test_result=analyze_text(TEST_TEXT,threshold=threshold_to_use)

## Word Contribution (“Weight ของคำ”)
ใช้วิธีเอาคำออกทีละคำแล้ววัด `Δ Confidence` ของทุกหมวด  
ค่า **บวก** = คำนั้นสนับสนุนหมวด / ค่า **ลบ** = คำนั้นผลักคะแนนออกจากหมวด  
นี่เป็น local explanation ตามบริบท ไม่ใช่ weight ภายใน Transformer โดยตรง

In [ ]:
# ============================================================
# CELL 32 : Word Contribution Functions
# ============================================================
from pythainlp.tokenize import word_tokenize

def word_contribution(text,max_words=30):
    base=score_dataframe(get_raw_scores(text)); base_probs={int(r.Label):float(r.Confidence) for _,r in base.iterrows()}; top_label=int(base.iloc[0].Label); top_cat=base.iloc[0].Category
    toks=word_tokenize(text,engine="newmm",keep_whitespace=True); idx=[i for i,t in enumerate(toks) if t.strip()][:max_words]; rows=[]
    for n,i in enumerate(idx,1):
        token=toks[i]; modified="".join(toks[:i]+toks[i+1:]).strip()
        if not modified: continue
        s=score_dataframe(get_raw_scores(modified)); probs={int(r.Label):float(r.Confidence) for _,r in s.iterrows()}; row={"ตำแหน่ง":i,"คำ":token,"ข้อความหลังเอาคำออก":modified}
        for label in LABEL_ORDER: row[f"Δ {CATEGORY_MAP[label]} (pp)"]=(base_probs[label]-probs[label])*100
        row["ผลต่อ Top-1 (pp)"]=(base_probs[top_label]-probs[top_label])*100; rows.append(row); print(f"{n}/{len(idx)}",end="\r")
    out=pd.DataFrame(rows)
    if len(out):
        den=out["ผลต่อ Top-1 (pp)"].abs().sum(); out["สัดส่วนอิทธิพลต่อ Top-1 (%)"]=out["ผลต่อ Top-1 (pp)"].abs()/den*100 if den>0 else 0; out=out.sort_values("ผลต่อ Top-1 (pp)",ascending=False).reset_index(drop=True)
    print("\nTop-1 เดิม:",top_cat,"Confidence",f"{base_probs[top_label]*100:.2f}%")
    return out,base

In [ ]:
# ============================================================
# CELL 33 : ⭐ ดู Weight/Contribution ของแต่ละคำ
# ============================================================
WORD_TEST_TEXT="รถเมย์ชนหมาเลือดสาดแม่บ้านมาทำความสะอาดหน่อย"
word_df,base_score_df=word_contribution(WORD_TEST_TEXT,max_words=30)
display(word_df[["คำ","ผลต่อ Top-1 (pp)","สัดส่วนอิทธิพลต่อ Top-1 (%)"]].round(3))
print("ตารางเต็ม: คำ x ทุกหมวด")
display(word_df.round(3))

In [ ]:
# ============================================================
# CELL 34 : Word Contribution Plot
# ============================================================
if len(word_df):
    p=word_df.sort_values("ผลต่อ Top-1 (pp)").tail(15); plt.figure(figsize=(10,6)); plt.barh(p["คำ"],p["ผลต่อ Top-1 (pp)"]); plt.axvline(0,linewidth=1); plt.xlabel("Δ Confidence Top-1 (percentage points)"); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# CELL 35 : ⭐ Interactive Threshold UI 1% - 99%
# ============================================================
import ipywidgets as widgets
from IPython.display import display,clear_output
text_widget=widgets.Textarea(value="รถเมย์ชนหมาเลือดสาดแม่บ้านมาทำความสะอาดหน่อย",description="ข้อความ:",layout=widgets.Layout(width="95%",height="100px"))
threshold_widget=widgets.FloatSlider(value=float(AUTO_THRESHOLD),min=.01,max=.99,step=.01,description="Threshold",readout_format=".2f",continuous_update=False)
margin_widget=widgets.FloatSlider(value=MARGIN_THRESHOLD,min=0,max=.50,step=.01,description="Margin",readout_format=".2f",continuous_update=False)
button=widgets.Button(description="วิเคราะห์",button_style="primary"); out=widgets.Output()
def click(_):
    with out:
        clear_output(wait=True); analyze_text(text_widget.value,threshold=float(threshold_widget.value),margin_threshold=float(margin_widget.value))
button.on_click(click); display(widgets.VBox([text_widget,threshold_widget,margin_widget,button,out]))

In [ ]:
# ============================================================
# CELL 36 : Error Analysis
# ============================================================
err=test_pred_df.copy(); err["true_category"]=err.true_label.map(CATEGORY_MAP); err["pred_category"]=err.pred_label.map(CATEGORY_MAP); err["correct"]=err.true_label==err.pred_label; err["accepted"]=(err.confidence>=AUTO_THRESHOLD)&(err.margin>=MARGIN_THRESHOLD)
print("ทายผิด"); display(err[~err.correct].sort_values("confidence",ascending=False))
print("\nถูก Reject/Confidence ต่ำ"); display(err[~err.accepted].sort_values("confidence"))

In [ ]:
# ============================================================
# CELL 37 : Save Evaluation Results
# ============================================================
metrics_summary.to_csv(EVAL_DIR/"test_metrics_summary.csv",index=False,encoding="utf-8-sig")
threshold_df.to_csv(EVAL_DIR/"threshold_search.csv",index=False,encoding="utf-8-sig")
validation_pred_df.to_csv(EVAL_DIR/"validation_predictions.csv",index=False,encoding="utf-8-sig")
test_pred_df.to_csv(EVAL_DIR/"test_predictions.csv",index=False,encoding="utf-8-sig")
err.to_csv(EVAL_DIR/"error_analysis.csv",index=False,encoding="utf-8-sig")
print("บันทึกผลที่",EVAL_DIR)

# สรุปการแปลผล
- **Training Loss**: บอกการเรียนรู้บน Train; ต่ำไม่ได้แปลว่า Accuracy สูง
- **Validation Loss**: ใช้จับ Overfitting และเลือก checkpoint
- **Macro-F1**: Metric หลักที่แนะนำเมื่อแต่ละหมวดมีจำนวนไม่เท่ากัน
- **Confusion Matrix**: หา class ที่สับสนกัน เช่น อาคาร–IT หรือ รถเมล์–ความปลอดภัย
- **Threshold**: เลือกจาก Validation ด้วย Coverage vs Selective Accuracy ไม่ใช่เดา 75/80%
- **Margin**: Top-1 และ Top-2 ใกล้กัน = ความกำกวมสูง
- **Temperature Scaling**: ปรับ confidence ให้สมเหตุสมผลขึ้น
- **Word Contribution**: ใช้อธิบาย “คำไหนผลักการตัดสินใจ” แบบ local explanation
- **Multi-Issue**: Top-2/Margin เป็นเพียง heuristic เพราะโมเดลนี้ single-label; ถ้าต้องส่งหลายหน่วยงานจริงควรพัฒนา multi-label หรือ complaint decomposition เพิ่ม

## ข้อเสนอแนะเมื่อเพิ่ม Dataset
1. เก็บข้อความภาษาไทยจริงของมหาวิทยาลัย
2. เพิ่มข้อมูลให้ทุก class เพียงพอ
3. เก็บ `Complaint_Group_ID` ก่อน paraphrase/augmentation
4. แยก group ก่อนทำ augmentation เพื่อป้องกัน leakage
5. ใช้ Test set เฉพาะรายงานผลสุดท้าย
6. เปรียบเทียบ Baseline, Encoder classifier (เช่น WangchanBERTa/XLM-R) และ LLM QLoRA